# Загрузка датасета Santander Product Recommendation

Данные берутся из соревнования Kaggle **[Santander Product Recommendation](https://www.kaggle.com/competitions/santander-product-recommendation/data)**
(`santander-product-recommendation`).

Ноутбук делает три вещи:
1. Проверяет, что настроены ключи Kaggle API (`.env` → `KAGGLE_USERNAME`/`KAGGLE_KEY`
   или `~/.kaggle/kaggle.json`).
2. Скачивает `train_ver2.csv` (≈2.3 ГБ, 13 647 309 строк, 48 колонок) в каталог `data/`.
3. Проверяет целостность скачанного файла: размер, заголовок, число строк.

> **Перед первым запуском:** на вкладке *Rules* страницы соревнования нужно нажать
> *I Understand and Accept*, иначе Kaggle API вернёт 403. Ключ API создаётся в
> *Kaggle → Settings → API → Create New Token* (`kaggle.json`), значения `username`/`key`
> из него кладутся в `.env` (шаблон — `.env.example`).

Датасет в репозиторий не коммитится: каталог `data/` указан в `.gitignore`.

In [ ]:
import os
import zipfile
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Ключи Kaggle и прочие секреты — в .env (см. .env.example)
load_dotenv()

COMPETITION = os.getenv('KAGGLE_COMPETITION', 'santander-product-recommendation')
DATA_DIR = Path(os.getenv('DATA_DIR', 'data'))
TRAIN_FILE = 'train_ver2.csv'
FORCE_DOWNLOAD = os.getenv('FORCE_DOWNLOAD', '0') == '1'
COUNT_ROWS = os.getenv('COUNT_ROWS', '1') == '1'

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'Соревнование: {COMPETITION}')
print(f'Каталог данных: {DATA_DIR.resolve()}')
print(f'Целевой файл: {DATA_DIR / TRAIN_FILE}')
print(f'Перекачивать принудительно: {FORCE_DOWNLOAD}')

In [ ]:
def check_kaggle_credentials() -> None:
    """Проверяет, что ключи Kaggle доступны, и объясняет, что делать, если их нет."""
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    env_ready = bool(os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'))

    if kaggle_json.exists():
        # Kaggle CLI требует права 600 на файл с ключом
        kaggle_json.chmod(0o600)

    if not env_ready and not kaggle_json.exists():
        raise RuntimeError(
            'Не найдены ключи Kaggle API.\n'
            '1) Kaggle → Settings → API → Create New Token — это скачает kaggle.json;\n'
            '2) положите kaggle.json в ~/.kaggle/kaggle.json (chmod 600) или скопируйте '
            'username/key в .env как KAGGLE_USERNAME и KAGGLE_KEY (см. .env.example);\n'
            '3) примите правила соревнования: '
            f'https://www.kaggle.com/competitions/{COMPETITION}/rules'
        )

    print('Ключи Kaggle найдены:', 'environment/.env' if env_ready else str(kaggle_json))


check_kaggle_credentials()

In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()


def extract_train_file(archive: Path) -> None:
    """Достаёт train_ver2.csv из zip-архива, скачанного Kaggle API."""
    with zipfile.ZipFile(archive) as zf:
        names = zf.namelist()
        if TRAIN_FILE not in names:
            raise FileNotFoundError(
                f'В архиве {archive.name} нет {TRAIN_FILE}. Содержимое: {names}'
            )
        print(f'Распаковываю {TRAIN_FILE} из {archive.name}...')
        zf.extract(TRAIN_FILE, path=DATA_DIR)


def download_train_file() -> Path:
    """Скачивает train_ver2.csv в data/ и возвращает путь к файлу.

    Сначала пробуем скачать один файл (это быстрее), при неудаче — весь архив
    соревнования. Kaggle API может отдать как готовый csv, так и zip.
    """
    target = DATA_DIR / TRAIN_FILE

    if target.exists() and not FORCE_DOWNLOAD:
        print(f'{target} уже на месте — скачивание пропущено '
              f'(нужно перекачать: FORCE_DOWNLOAD=1).')
        return target

    try:
        print('Скачиваю train_ver2.csv...')
        api.competition_download_file(
            COMPETITION, TRAIN_FILE, path=str(DATA_DIR), force=True, quiet=False
        )
    except Exception as exc:  # нет доступа/файл отдаётся только в общем архиве
        print(f'Не удалось скачать отдельный файл ({exc}).\n'
              'Качаю полный архив соревнования...')
        api.competition_download_files(COMPETITION, path=str(DATA_DIR), force=True, quiet=False)

    if target.exists():
        return target

    archives = sorted(DATA_DIR.glob('*.zip'))
    if not archives:
        raise FileNotFoundError(
            f'После скачивания в {DATA_DIR} нет ни {TRAIN_FILE}, ни zip-архива. '
            'Проверьте, что правила соревнования приняты и ключи Kaggle верны.'
        )
    extract_train_file(archives[-1])
    return target


train_path = download_train_file()
print(f'Готово: {train_path}')

In [ ]:
expected_columns = [
    'fecha_dato', 'ncodpers', 'ind_empleado', 'pais_residencia', 'sexo', 'age',
    'fecha_alta', 'ind_nuevo', 'antiguedad', 'indrel', 'ult_fec_cli_1t',
    'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'conyuemp', 'canal_entrada',
    'indfall', 'tipodom', 'cod_prov', 'nomprov', 'ind_actividad_cliente', 'renta',
    'segmento', 'ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1',
    'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1',
    'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1',
    'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1',
    'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1',
    'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1',
    'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1',
    'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1',
]

size_gb = train_path.stat().st_size / 1024 ** 3
head = pd.read_csv(train_path, nrows=5)

missing_columns = sorted(set(expected_columns) - set(head.columns))
extra_columns = sorted(set(head.columns) - set(expected_columns))

print(f'Размер файла: {size_gb:.2f} ГБ')
print(f'Колонок: {len(head.columns)} (ожидалось {len(expected_columns)})')
if missing_columns:
    print(f'ВНИМАНИЕ, нет колонок: {missing_columns}')
if extra_columns:
    print(f'ВНИМАНИЕ, лишние колонки: {extra_columns}')
display(head)

if COUNT_ROWS:
    n_rows = sum(1 for _ in open(train_path, 'rb'))
    print(f'Строк в файле (с заголовком): {n_rows:,}')
    print('Ожидалось ~13 647 309 строк данных — расхождение допустимо, '
          'если ранее датасет уже фильтровался.')
else:
    print('Подсчёт строк пропущен (COUNT_ROWS=0).')

## Что дальше

Файл `data/train_ver2.csv` — исходные данные для всего пайплайна, их читают:

* `eda.ipynb` — исследовательский анализ;
* `modeling.ipynb` — предобработка, ALS-признак, обучение и тюнинг модели;
* `rec_sys.ipynb`, `test.ipynb` — офлайн-проверка инференса и нагрузочный прогон сервиса.

Файлы соревнования `test_ver2.csv` и `sample_submission.csv` для этого проекта не нужны:
задача решается как cross-sell по истории 2015 года с проверкой на 2016 год,
то есть отложенная выборка строится из `train_ver2.csv` (см. `modeling.ipynb`, раздел про целевую переменную).